In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import glob, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 22})
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
import matplotlib.patches as mpatches
import pandas as pd

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
def global_average(dset):     
    lat_dim = 'lat' if 'lat' in dset.dims else 'latitude'
    lon_dim = 'lon' if 'lon' in dset.dims else 'longitude'
    
    weights = np.cos(np.deg2rad(dset[lat_dim]))
    weights.name = "weights"
    
    return dset.weighted(weights).mean((lat_dim, lon_dim))

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
def main_script(var, era_var, dset_ctrl, dset_sens, era, save_path, lead, season):
    """Main processing script for computing model biases against ERA5."""
    logging.info(f"Starting main {lead}")
    try:        
        # Compute global mean
        dset_ctrl_global = global_average(dset_ctrl[var]).to_dataset(name = var)
        dset_sens_global = global_average(dset_sens[var]).to_dataset(name = var)
        era_global = global_average(era[era_var]).to_dataset(name = var)

        #compute land mean
        dset_ctrl_land = global_average(af.land_seas_mask(dset_ctrl[var], 'land')).to_dataset(name = var)
        dset_sens_land = global_average(af.land_seas_mask(dset_sens[var], 'land')).to_dataset(name = var)
        era_land = global_average(af.land_seas_mask(era[era_var], 'land')).to_dataset(name = var)

        #compute ocean mean
        dset_ctrl_ocean = global_average(af.land_seas_mask(dset_ctrl[var], 'ocean')).to_dataset(name = var)
        dset_sens_ocean = global_average(af.land_seas_mask(dset_sens[var], 'ocean')).to_dataset(name = var)
        era_ocean = global_average(af.land_seas_mask(era[era_var], 'ocean')).to_dataset(name = var)
        
        # Compute time-mean biases
        bias_ctrl = dset_ctrl_global.mean('time', skipna=True) - era_global.mean('time', skipna=True)
        bias_sens = dset_sens_global.mean('time', skipna=True) - era_global.mean('time', skipna=True)

        bias_ctrl_land = dset_ctrl_land.mean('time', skipna=True) - era_land.mean('time', skipna=True)
        bias_sens_land = dset_sens_land.mean('time', skipna=True) - era_land.mean('time', skipna=True)
        
        bias_ctrl_ocean = dset_ctrl_ocean.mean('time', skipna=True) - era_ocean.mean('time', skipna=True)
        bias_sens_ocean = dset_sens_ocean.mean('time', skipna=True) - era_ocean.mean('time', skipna=True)
        

        # Assign member coordinates based on actual data size
        n_members = dset_ctrl.sizes['member']
        member_ids = xr.DataArray(np.arange(1, n_members + 1), dims='member', name='member')
        bias_ctrl['member'] = member_ids
        bias_sens['member'] = member_ids
        bias_ctrl_land['member'] = member_ids
        bias_sens_land['member'] = member_ids
        bias_ctrl_ocean['member'] = member_ids
        bias_sens_ocean['member'] = member_ids
        
        # Save results
        ctrl_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_average_all_members_{season}_1999.nc'
        sens_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_average_all_members_{season}_1999.nc'
        ctrl_outfile_land = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_average_all_members_land_{season}_1999.nc'
        sens_outfile_land = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_average_all_members_land_{season}_1999.nc'
        ctrl_outfile_ocean = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_average_all_members_ocean_{season}_1999.nc'
        sens_outfile_ocean = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_average_all_members_ocean_{season}_1999.nc'
        
        bias_ctrl.to_netcdf(ctrl_outfile)
        bias_sens.to_netcdf(sens_outfile)
        bias_ctrl_land.to_netcdf(ctrl_outfile_land)
        bias_sens_land.to_netcdf(sens_outfile_land)        
        bias_ctrl_ocean.to_netcdf(ctrl_outfile_ocean)
        bias_sens_ocean.to_netcdf(sens_outfile_ocean)
        
        logging.info(f"Bias files written successfully for lead {lead}")

    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
def run_main_script_for_season_and_lead(save_path, season, lead):
    logging.info(f"Starting lead {lead}")
    try:
        # Load control experiment dataset
        ctrl_path = f"{save_path}/{exp_ctrl}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_lead{lead}_rad_{season}.nc"
        dset_ctrl = xr.open_mfdataset(ctrl_path, concat_dim='member', combine='nested', chunks='auto').load()
        dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).to_period('M').to_timestamp()
    
        # Load sensitivity experiment dataset
        sens_path = f"{save_path}/{exp_sens}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_lead{lead}_rad_{season}.nc"
        dset_sens = xr.open_mfdataset(sens_path, concat_dim='member', combine='nested', chunks='auto').load()
        dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).to_period('M').to_timestamp()
        
        # Load ERA5 for the specific season inside the process
        era_path = f"{WORK_DIR}/ERA5_{era_var}_1x1_{season}.nc"
        era = xr.open_dataset(era_path, chunks='auto').load()
        era = era.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))

        #select time from 1999
        start_date = '1999-09-01'
        
        dset_ctrl = dset_ctrl.sel(time=slice(start_date, None))
        dset_sens = dset_sens.sel(time=slice(start_date, None))
        era = era.sel(time=slice(start_date, None))
        
        main_script(var, era_var, dset_ctrl, dset_sens, era, SAVE_PATH, lead, season)
    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
%%time
seasons = ['DJF', 'MAM', 'JJA', 'SON']
futures = []

# Submit all (season, lead) combinations in parallel
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [
        executor.submit(run_main_script_for_season_and_lead, SAVE_PATH, season, lead)
        for season in seasons
        for lead in leads
    ]
    concurrent.futures.wait(futures)